# least popular heros in the marvel graph

In [ ]:
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName('Superheros').getOrCreate()

# making the schema
schema = StructType([
    StructField('id', IntegerType()),
    StructField('name', StringType())
])

# reading in both the files needed
names = spark.read.schema(schema).option('sep', ' ').csv('../../MarvelNames.txt')
lines = spark.read.text('../../MarvelGraph.txt')


# grabbing the lines dataframe and returning each id with number of connections
connections = lines.withColumn('id', F.split(F.col('value'), ' ')[0]) \
.withColumn('connections', F.size(F.split(F.col('value'), ' ')) - 1) \
.groupBy('id').agg(F.sum('connections').alias('connections')) # groups all the id and counts total number of connections

# get the what the number is for the least connections
leastPopAmount = connections.agg(F.min(F.col('connections'))).first()[0]

# get all the ids with that connection that is the least
leastPopularTable = connections.filter(F.col('connections') == leastPopAmount)

# join the leastPopularTable with the name table in order to get the name of each id
nameJoin = leastPopularTable.join(names, on='id', how='inner')

# only select the name and the connections for the table being shown and sort alphabetically
nameJoin.select('name', 'connections').sort('name', ascending = True).show()

+--------------------+-----------+
|                name|connections|
+--------------------+-----------+
|        BERSERKER II|          1|
|              BLARE/|          1|
|     CALLAHAN, DANNY|          1|
|       CLUMSY FOULUP|          1|
|         DEATHCHARGE|          1|
|              FENRIS|          1|
|GERVASE, LADY ALYSSA|          1|
|      GIURESCU, RADU|          1|
|JOHNSON, LYNDON BAIN|          1|
|                KULL|          1|
|          LUNATIK II|          1|
|MARVEL BOY II/MARTIN|          1|
|MARVEL BOY/MARTIN BU|          1|
|              RANDAK|          1|
|         RED WOLF II|          1|
|                RUNE|          1|
|         SEA LEOPARD|          1|
|           SHARKSKIN|          1|
|              ZANTOR|          1|
+--------------------+-----------+

